# 01 · Export intensity and brightfield images

For every acquisition under `raw/<session>/<sample>/`:

* the **intensity image** is reconstructed from the `.ptu` file: photon
  counts summed over both detectors and all z planes, saved as 16-bit TIFF
  in `intensity/<session>/<sample>_export/`;
* the **brightfield z-stack** is read from the matching `.msr` file, where
  one was recorded, and saved in `intensity/<session>/<sample>_bf/`.

These images are the input of the segmentation notebooks (02a-c).

In [ ]:
import os

import numpy as np
import pandas as pd
from skimage import io

from tttrkit.ptuio.reader import TTTRReader
from tttrkit.ptuio.reconstructor import ImageReconstructor

import fret_flim_data_processing.config as config
from fret_flim_data_processing.helpers import (
    index_samples,
    make_scan_config,
    read_acquisition_params,
    read_msr_transmission,
    run_reconstruction,
    scan_shape,
    show_image_grid,
)

## Inputs

In [ ]:
raw_dir       = config.RAW_DIR         # <session>/<sample>/*.ptu and *.msr
intensity_dir = config.INTENSITY_DIR   # output

sessions = None   # e.g. ['2025-07-30']; None exports every session

frames                 = 5   # z planes per acquisition
fallback_accumulations = 4   # line accumulations assumed for files without a .msr

## Intensity images

The scan geometry differs between acquisitions (most are 100 x 100 px, some
are not), so it is read for each file: the line length from the `.ptu`
header and the number of lines from the `.msr` file. Without a `.msr`,
`fallback_accumulations` is assumed; those files are listed below.

In [ ]:
samples = {sample: info for sample, info in index_samples(raw_dir).items()
           if sessions is None or info['session'] in sessions}

intensity = {}   # sample -> {image base name -> intensity image}
rows = []

for sample, info in samples.items():
    export_dir = os.path.join(intensity_dir, info['session'], sample + '_export')
    os.makedirs(export_dir, exist_ok=True)
    intensity[sample] = {}

    for ptu_file in info['files']:
        base = os.path.splitext(ptu_file)[0]
        path = os.path.join(info['path'], ptu_file)

        reader = TTTRReader(path)
        tags   = reader.header.tags
        params = read_acquisition_params(tags)
        lines, pixels, source = scan_shape(path, tags, frames, fallback_accumulations)
        row = {'session': info['session'], 'sample': sample, 'file': ptu_file,
               'lines': lines, 'pixels': pixels, 'geometry_from': source}

        try:
            reconstructor = ImageReconstructor(
                config=make_scan_config(tags, lines, pixels, frames),
                tcspc_channels=params['tcspc_bins'],
                outputs=['photon_count'],
            )
            result = run_reconstruction(reader, reconstructor, params['wrap'])
        except Exception as error:
            rows.append({**row, 'exported': False, 'note': repr(error)})
            continue

        image = np.asarray(result.photon_count.sum(dim=['frame', 'channel', 'sequence']))
        io.imsave(os.path.join(export_dir, base + '.tif'), image.astype(np.uint16),
                  check_contrast=False)

        intensity[sample][base] = image
        rows.append({**row, 'exported': True, 'note': ''})

export_log = pd.DataFrame(rows)
print(f"Exported {int(export_log['exported'].sum())} of {len(export_log)} image(s) "
      f"from {len(samples)} sample(s) to {intensity_dir}")

# Worth a look: failed files, and geometries that were assumed rather than read.
export_log[~export_log['exported'] | (export_log['geometry_from'] != '.msr')]

In [ ]:
for sample, sample_images in intensity.items():
    show_image_grid(list(sample_images.values()), f"{sample} - intensity")

## Brightfield stacks

The transmitted-light detector is not recorded in the `.ptu`; its image is
stored only in the `.msr` file, on the same scan as the FLIM z-stack. The full
z-stack is saved rather than a projection, because brightfield contrast
inverts through focus and a mean over the planes cancels most of it.

In [ ]:
brightfield = {}   # sample -> {image base name -> (z, y, x) stack}
missing = []

for sample, info in samples.items():
    bf_dir = os.path.join(intensity_dir, info['session'], sample + '_bf')
    brightfield[sample] = {}

    for base, image in intensity[sample].items():
        msr_path = os.path.join(info['path'], base + '.msr')
        try:
            stack = read_msr_transmission(msr_path)
        except (OSError, KeyError, ValueError) as error:
            missing.append(f"{sample}/{base}: {error}")
            continue

        if stack.min() < 0:
            raise ValueError(f"{msr_path}: negative transmission counts ({stack.min()})")
        if stack.shape[1:] != image.shape:
            print(f"WARNING {sample}/{base}: brightfield is {stack.shape[1:]}, "
                  f"intensity image is {image.shape}")

        os.makedirs(bf_dir, exist_ok=True)
        io.imsave(os.path.join(bf_dir, base + '.tif'), stack.astype(np.uint16),
                  check_contrast=False)
        brightfield[sample][base] = stack

print(f"Exported {sum(len(v) for v in brightfield.values())} brightfield stack(s); "
      f"{len(missing)} image(s) have none.")

Preview of the standard deviation over z, the projection used for cell
segmentation in notebook 02a.

In [ ]:
for sample, stacks in brightfield.items():
    if stacks:
        show_image_grid([stack.astype(np.float32).std(axis=0) for stack in stacks.values()],
                        f"{sample} - brightfield (SD over z)")